# Pandera Exploration

In [ ]:
import pandas as pd
import pandera.pandas as pa             # när man jobbar med pandas DataFrames är det mycket rekommenderat att använda pandera.pandas istället för bara pandera

### DataFrameSchema

In [ ]:
# data att validera
df = pd.DataFrame({
    "station": ["ST1", "ST1", "ST3", "ST5"],
    "kunder": [1000, 400, 84, 2000],
    "avbrottstyp": ["planerat", "oplanerat", "oplanerat", "planerat"]
})

print(df)

In [ ]:
# sätt upp schema
schema = pa.DataFrameSchema({
    "station": pa.Column(
        str,
        pa.Check.isin(["ST1", "ST2", "ST3", "ST4", "ST5"])
    ),
    "kunder": pa.Column(int, pa.Check.ge(0)),
    "avbrottstyp": pa.Column(
        str,
        pa.Check.isin(["planerat", "oplanerat"])
    )
})

In [ ]:
validated_df = schema.validate(df)
print(validated_df)

Då datan stämde returneras df:en utan fel. 

In [ ]:
# data med fel 
df_fel = pd.DataFrame({
    "station": ["ST1", "ST1", "ST3", "ST6"],
    "kunder": [1000, 400, 84, 2000.4],
    "avbrottstyp": ["planerat", "oplanerat", "oplanerat", "planerat"]
})

print(df_fel)

In [ ]:
# validated_df_fel = schema.validate(df_fel)

Koden kraschar med ett SchemaError vid första felet (ST6 är inte en godkänd station)

### Dataframe Model

In [ ]:
# Definiera schema typ som en dataclass
class Schema(pa.DataFrameModel):
    station: str = pa.Field(isin=["ST1", "ST2", "ST3", "ST4", "ST5"])
    kunder: int = pa.Field(ge=0)
    avbrottstyp: str = pa.Field(isin=["planerat", "oplanerat"])

Schema.validate(df)

In [ ]:
# Test med felaktig data
# Schema.validate(df_fel)

Och det blir mycket riktigt ett SchemaError vid första felet.

### Informativa fel

In [ ]:
simple_schema = pa.DataFrameSchema({
    "voltage_level_kv": pa.Column(
        float,
        pa.Check(
            lambda x: 0.0 <= x <= 20.0,
            element_wise=True,
            error="range checker [0, 20]"
        )
    )
})

# datan bryter mot regeln
fail_check_df = pd.DataFrame({
    "voltage_level_kv": [-4.0, 0.4, 10.0, 20.0]
})

try:
    simple_schema(fail_check_df)
except pa.errors.SchemaError as exc:
    print(exc)

In [ ]:
# Med felaktigt kolumnnamn
wrong_column_df = pd.DataFrame({
    "duration_minutes": [5.6, 9.0]
})

try:
    simple_schema(wrong_column_df)
except pa.errors.SchemaError as exc:
    print(exc)

### Felrapporter

In [ ]:
# data med fel 
df_fel = pd.DataFrame({
    "station": ["ST1", "ST1", "ST3", "ST6"],
    "kunder": [1000, 400, 84, 2000.4],
    "avbrottstyp": ["planerat", "oplanerat", "oplanerat", "planerat"]
})

print(df_fel)

In [ ]:
# sätt upp schema
schema = pa.DataFrameSchema({
    "station": pa.Column(
        str,
        pa.Check.isin(["ST1", "ST2", "ST3", "ST4", "ST5"])
    ),
    "kunder": pa.Column(int, pa.Check.ge(0)),
    "avbrottstyp": pa.Column(
        str,
        pa.Check.isin(["planerat", "oplanerat"])
    )
},
name="MySchema",
strict=True                                     # strict=True --> "a column in the dataframe is not specified in the schema", behövs inte för min exempeldata
)

try:
    schema.validate(df_fel, lazy=True)
except pa.errors.SchemaErrors as exc:
    print(exc)

"DATAFRAME_CHECK" visar vad det är som är fel (dvs ST6 finns inte med som godkänd station), men WRONG_DATATYPE visar inte vad som är fel (dvs 2000.4 är en float och inte en int). 

Vad beror detta på?

Check.isin(["ST1", "ST2", "ST3", "ST4", "ST5"]) kollar specifikt efter dessa strängar, medan Column(int, pa.Check.ge(0)) bara kollar att hela raden/serien är int. 

Hela kolumnen ser ut att ha blivit float64.

In [ ]:
print(df_fel.dtypes)

Just det, Series är ndarray-like (https://pandas.pydata.org/docs/user_guide/dsintro.html#series-is-ndarray-like) och numpy.ndarray är homogen (https://numpy.org/doc/stable/reference/generated/numpy.ndarray.html#numpy.ndarray) vilket innebär att en float gör hela serien till float. 

Hur gör man för att fånga upp detta typ av fel?

In [ ]:
# Exempel 

df = pd.DataFrame({"kunder": [1000, 400, 84, 2000.4]})

schema = pa.DataFrameSchema({"kunder": pa.Column(int)})

try:
    schema.validate(df)
except pa.errors.SchemaError as exc:
    print(exc)

https://pandera.readthedocs.io/en/stable/dtype_validation.html#data-type-coercion

-->

https://pandera.readthedocs.io/en/stable/parsers.html 

-->

https://pandera.readthedocs.io/en/stable/dataframe_schemas.html#coerced

Om det smyger sig in 2000.4 i kolumnen "kunder" så kan det ju innebära att hela talet är fel, inte bara att decimalen är fel. Det kanske till exempel skulle vara 20004.

Därför kan det bli dumt att med `coerced=True` göra så att 2000.4 blir en int (dvs 2000). 

Modulusoperatorn (%) ger det som blir över efter division mellan två tal:

In [ ]:
print(2000.4 % 1)
print(2000.4 % 2)
print(2000.4 % 3)

Med hjälp av modulus går det alltså att kontrollera om decimalen är 0 eller något annat tal:

In [ ]:
2000.4 % 1 == 0

In [ ]:
2000.0 % 1 == 0

https://pandera.readthedocs.io/en/stable/checks.html#registering-custom-checks 

--> 

https://pandera.readthedocs.io/en/stable/extensions.html#registering-custom-check-methods 

Men frågan är om det inte är mest korrekt att låta allt krascha om indatan innehåller floats trots att det inte är tillåtet. 

(Och vid krasch köra en separat kontroll för att se vad det är som har blivit fel så det går att fixa)

### Tänkt datamodell

| Kolumnnamn | Dtype | Validering |
| --- | --- | --- |
| incident_id | str | Kolla unique |
| voltage_level_kv | float | Tillåtna värden 0,4, 10.0 och 20.0 |
| start_time | datetime | Giltigt tidsintervall |
| end_time | datetime | Måste vara efter start_time |
| duration_minutes | int | Kontroll mot start_time och end_time |
| outage_type | str | Kontroll mot fasta kategorier | 
| cause_category | str | Kontroll mot fasta kategorier samt att om outage_type == "planerat" så kan inte cause_category vara t ex "väder" |
| customers_affected | int | Heltal, icke-negativt |
| compensation_eligible | bool | Ska vara True om duration_minutes >= 720 minuter och outage_type == "oplanerat" |


### Unique

Standard är att unique=False 

https://pandera.readthedocs.io/en/stable/reference/generated/pandera.api.pandas.components.Column.html#pandera.api.pandas.components.Column 

In [ ]:
df = pd.DataFrame({"incident_id": ["GBG-2026-001", "GBG-2026-002", "GBG-2026-003", "GBG-2026-003"]})

schema = pa.DataFrameSchema({
    "incident_id": pa.Column(
        str,
        unique=True
    )
})

In [ ]:
try:
    schema(df)
except pa.errors.SchemaError as exc:
    print(exc)

### Datetime | Timestamp

https://pandera.readthedocs.io/en/stable/reference/generated/pandera.dtypes.Timestamp.html#pandera.dtypes.Timestamp

Pandera kan omvandla till Timestamp.

In [ ]:
df_timestamp = pd.DataFrame({"start_time": ["2026-01-01 08:00:00", "2026-02-02 08:00:00", "2026-03-03 08:00:00"]})

In [ ]:
schema_timestamp = pa.DataFrameSchema({
    "start_time": pa.Column(
        pa.DateTime,
        coerce=True
    )
})

In [ ]:
try:
    schema_timestamp(df_timestamp)
except pa.errors.SchemaError as exc:
    print(exc)

In [ ]:
# Test med felaktig data
df_timestamp_fel = pd.DataFrame({"start_time": ["2026-01-01 08:00:00", "hej", "2026-03-03 08:00:00"]})

try:
    schema_timestamp(df_timestamp_fel)
except pa.errors.SchemaErrors as exc:
    print(exc)

In [ ]:
# Test med olika typer av datumformat
test_cases = [
    "2026-01-01 08:00:00",
    "2026-05-12",
    "2026/05/12 14:30:00",  
    "12 maj 14:30",  
    "14:30:00",         
    "2026-02-30 10:00:00",
    "2026-13-01 00:00:00",
    "hej",
    "260512-1430",
]

for val in test_cases:
    df_temp = pd.DataFrame({"start_time": [val]})
    try:
        res = schema_timestamp(df_temp)
        print(f"PASSED : {repr(val):<25} -> Tolkat till: {res['start_time'].iloc[0]}")
    except (pa.errors.SchemaError, pa.errors.SchemaErrors):
        print(f"FAILED : {repr(val):<25}")

Detta gör ju att väldigt mycket "konstiga" format godkänns med risk att analyser blir felaktiga. 

Ner i ett "rabbit hole" om datastädning vs datavalidering och hur ser data ut enligt standard egentligen när den kommer till valideringssteget... Vad ska Data Validation göra?? 

För att komma vidare behöver jag nog avgränsa mig ännu mer och då till ett valideringsflöde som kontrollerar att affärslogiken stämmer (business rule validation) och som inte innebär Fail-Fast.

### Wide Checks 

https://pandera.readthedocs.io/en/stable/checks.html#wide-checks 

In [ ]:
df_wide = pd.DataFrame({
    "outage_type": ["planerat", "planerat", "oplanerat"],
    "cause_category": ["underhåll", "väder", "underhåll"]
})

schema_wide = pa.DataFrameSchema(
    checks=pa.Check(
        lambda df: (df["cause_category"] == "underhåll") == (df["outage_type"] == "planerat"),
        error="Ogiltig kombination av typ och orsak"
    )
)

try:
    schema_wide.validate(df_wide)
except pa.errors.SchemaError as exc:
    print(exc)

### Lazy Validation

https://pandera.readthedocs.io/en/stable/lazy_validation.html

In [ ]:
import json

schema = pa.DataFrameSchema(
    columns={
        "outage_type": pa.Column(str, pa.Check.isin(["planerat", "oplanerat"])),
        "cause_category": pa.Column(str, pa.Check.isin(["underhåll", "väder", "tekniskt fel"])),
        "customers_affected": pa.Column(int, pa.Check.greater_than_or_equal_to(0))
    },
    strict=True,
    checks=[
        pa.Check(
            lambda df: (df["cause_category"] == "underhåll") == (df["outage_type"] == "planerat"),
            name="check_outage_cause_relation"
        )
    ]
)

df = pd.DataFrame({
    "outage_type": ["planerat", "oplanerat", "planerat", "oplanerat"],
    "cause_category": ["väder", "underhåll", "underhåll", "tekniskt fel"],
    "customers_affected": [-5, 10, 0, 20],
    "extra_kolumn": [1, 2, "a", "b"]                                        # pga strict=True
})

try:
    schema.validate(df, lazy=True)
except pa.errors.SchemaErrors as exc:
    print(json.dumps(exc.message, indent=2, default=str))

In [ ]:
try:
    schema.validate(df, lazy=True)
except pa.errors.SchemaErrors as exc:
    print("Schema errors and failure cases:")
    print(exc.failure_cases)
    print("\nDataFrame object that failed validation:")
    print(exc.data)

    exc.data.to_csv("../data_for_exploration/test_felaktig_data.csv", index=False)

In [ ]:
# Test utan felaktig extra kolumn
import json

schema = pa.DataFrameSchema(
    columns={
        "outage_type": pa.Column(str, pa.Check.isin(["planerat", "oplanerat"])),
        "cause_category": pa.Column(str, pa.Check.isin(["underhåll", "väder", "tekniskt fel"])),
        "customers_affected": pa.Column(int, pa.Check.greater_than_or_equal_to(0))
    },
    strict=True,
    checks=[
        pa.Check(
            lambda df: (df["cause_category"] == "underhåll") == (df["outage_type"] == "planerat"),
            name="check_outage_cause_relation"
        )
    ]
)

df = pd.DataFrame({
    "outage_type": ["planerat", "oplanerat", "planerat", "oplanerat"],
    "cause_category": ["väder", "underhåll", "underhåll", "tekniskt fel"],
    "customers_affected": [-5, 10, 0, 20]
})

try:
    schema.validate(df, lazy=True)
except pa.errors.SchemaErrors as exc:
    print(json.dumps(exc.message, indent=2, default=str))

In [ ]:
try:
    schema.validate(df, lazy=True)
except pa.errors.SchemaErrors as exc:
    print("Schema errors and failure cases:")
    display(exc.failure_cases)
    print("\nDataFrame object that failed validation:")
    print(exc.data)

    exc.data.to_csv("../data_for_exploration/test_felaktig_data_igen.csv", index=False)

Alla rader hamnar i DataFrame:n/csv-filen.

Rad 0 och 1 innehåller fel, men även rad 2 och 3 kommer med.

Korskontrollen, check_outage_cause_relation, görs även på kolumnen customers_affected, vilket inte är så konstigt när customers_affected ligger med i columns={}.

Men det är ju såklart exc.failure_cases jag vill spara. Testar igen:

In [ ]:
try:
    schema.validate(df, lazy=True)
except pa.errors.SchemaErrors as exc:
    print("Schema errors and failure cases:")
    display(exc.failure_cases)
    print("\nDataFrame object that failed validation:")
    print(exc.data)

    exc.failure_cases.to_csv("../data_for_exploration/failure_cases.csv", index=False)

Sortera index 

Testa att lägga till en till rad i df:en och se om det är det indexet som kommer upp? dvs 4

### Facit för staged_outages.csv

- 0 (INC-2026-001): Korrekt (men blir ju fel pga rad 23)
- 1 (INC-2026-018): Felaktig duration_minutes (-120)
- 2 (INC-2026-002): Korrekt
- 3 (INC-2026-015): Fel spänningsnivå (130.0 kV)
- 4 (INC-2026-003): Korrekt
- 5 (INC-2026-020): Otillåten orsak för planerat (väder)
- 6 (INC-2026-004): Korrekt
- 7 (INC-2026-005): Korrekt
- 8 (INC-2026-016): Framtida datum (2038)
- 9 (INC-2026-006): Korrekt
- 10 (INC-2026-022): Negativt antal kunder (-15)
- 11 (INC-2026-007): Korrekt
- 12 (INC-2026-019): Ogiltig kategori (akut)
- 13 (INC-2026-008): Korrekt
- 14 (INC-2026-024): Multipel-fel (130.0, grävskada för planerat, -50, felaktig ersättning)
- 15 (INC-2026-009): Korrekt
- 16 (INC-2026-021): Otillåten orsak för oplanerat (underhåll)
- 17 (INC-2026-010): Korrekt
- 18 (INC-2026-017): end_time före start_time
- 19 (INC-2026-011): Korrekt
- 20 (INC-2026-023): Ogiltig ersättning vid 120 min
- 21 (INC-2026-012): Korrekt
- 22 (INC-2026-013): Korrekt
- 23 (INC-2026-001): Dubblett-ID mot rad 0
- 24 (INC-2026-014): Korrekt


### Testa kod i valideringsflödet

In [3]:
from pathlib import Path
import sys
import pandas as pd
import pandera.pandas as pa

# Lägg till rotmappen i Python-sökvägen så src kan importeras
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

# Importera ditt nya schema
from src.schemas import outage_schema

# Läs in staged-datan
df_staged = pd.read_csv(PROJECT_ROOT / "data" / "staged_outages.csv")

# Validera med lazy=True för att fånga alla fel
try:
    validated_df = outage_schema.validate(df_staged, lazy=True)
    print("All data passerade utan fel!")
except pa.errors.SchemaErrors as exc:
    print(f"Hittade {len(exc.failure_cases)} felaktigheter fördelat på raderna:\n")
    print(exc.failure_cases[["check", "column", "index", "failure_case"]])

    failed_indices = sorted(exc.failure_cases["index"].dropna().astype(int).unique())
    print("Felande rader:", failed_indices)

Hittade 63 felaktigheter fördelat på raderna:

                                                check              column  \
31     Konflikt mellan outage_type och cause_category    duration_minutes   
47  Ogiltig ersättningsstatus i förhållande till d...    voltage_level_kv   
34     Konflikt mellan outage_type och cause_category         outage_type   
35     Konflikt mellan outage_type och cause_category         outage_type   
36     Konflikt mellan outage_type och cause_category      cause_category   
..                                                ...                 ...   
1                                    field_uniqueness         incident_id   
6                     isin(['planerat', 'oplanerat'])         outage_type   
7            customers_affected kan inte vara negativ  customers_affected   
8            customers_affected kan inte vara negativ  customers_affected   
0                                    field_uniqueness         incident_id   

    index  failure_case  
31